In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sksurv.ensemble import RandomSurvivalForest, ComponentwiseGradientBoostingSurvivalAnalysis, GradientBoostingSurvivalAnalysis
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.inspection import permutation_importance
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored,concordance_index_ipcw, integrated_brier_score,brier_score, cumulative_dynamic_auc
from sklearn.model_selection import train_test_split,  GridSearchCV, ShuffleSplit, StratifiedShuffleSplit
from sksurv.preprocessing import OneHotEncoder,StandardScaler
from sksurv.util import Surv
import scipy.stats as st
import warnings
warnings.filterwarnings('ignore')

def Get_landmarkingset(Long_set,landmarking_time):
    Static_set=pd.DataFrame(columns=Long_set.columns)
    Long_set=Long_set[Long_set.Time<=landmarking_time]
    for patid in Long_set['Patient ID'].unique():
        
        data=Long_set[Long_set['Patient ID']==patid]
        Static_set=pd.concat([Static_set,data.loc[data.index[(data.Time-landmarking_time).abs().argmin()]].to_frame().T],ignore_index=True)       
    return Static_set



def mean_confidence_interval3(data):
    return round(np.mean(data),2), round(statistics.stdev(data),2) #, round(np.std(data,ddof=1),3)
from sklearn.linear_model import Lasso

def Feature_importance(X_train,y_train,n_folds=10,model_flag=False,model=[]):
    df_results=pd.DataFrame(columns=X_train.columns.tolist())
    for j in range(0,n_folds):
            X_t, X_te, y_t, y_te= train_test_split(X_train,y_train,test_size=1/n_folds, train_size=None)
            for i in X_t.columns:
                if (X_t[i] == X_t[i].iloc[0]).all():
                    print(i)
                    X_t=X_t.drop(i,axis=1)
                    X_te=X_te.drop(i,axis=1)                   
            if model_flag:
                reg=model
            else:
                reg = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.1) #model
            reg.fit(X_t, y_t)
            result = permutation_importance(reg, X_te, y_te, n_repeats=15)
            df_results.loc[len(df_results),X_t.columns]=result["importances_mean"]
    return df_results.sum().sort_values(ascending=False) 


def performance_metrics(estimator,y_train,y_test,times,X_train,X_test):
    cox_chf_funcs=estimator.predict_survival_function(X_test, return_array=False)
    cox_surv_prob=np.vstack([chf(times) for chf in cox_chf_funcs])
    cox_chf_funcs=estimator.predict_cumulative_hazard_function(X_test, return_array=False)
    cox_risk_scores=np.vstack([chf(times) for chf in cox_chf_funcs])
    c_index_train=estimator.score(X_train, y_train)
    c_index_test_ipcw=concordance_index_ipcw(y_train,y_test,estimator.predict(X_test))[0]
    c_index_test_ipcw0=concordance_index_ipcw(y_train,y_test,estimator.predict(X_test),tau=times[0])[0]
    c_index_test_ipcw1=concordance_index_ipcw(y_train,y_test,estimator.predict(X_test),tau=times[1])[0]
    c_index_test_ipcw2=concordance_index_ipcw(y_train,y_test,estimator.predict(X_test),tau=times[2])[0]
    c_index_test_ipcw3=concordance_index_ipcw(y_train,y_test,estimator.predict(X_test),tau=times[3])[0]
    cum_auc=cumulative_dynamic_auc(y_train,y_test,cox_risk_scores, times=times)
    brier_score_cox=brier_score(y_train, y_test,cox_surv_prob, times=times)
    brier_score_one=integrated_brier_score(y_train, y_test,cox_surv_prob,times=times)
    return c_index_train, c_index_test_ipcw,c_index_test_ipcw0, c_index_test_ipcw1, c_index_test_ipcw2, c_index_test_ipcw3,cum_auc[0][0],cum_auc[0][1],cum_auc[0][2],cum_auc[0][3],brier_score_cox[1][0],brier_score_cox[1][1],brier_score_cox[1][2],brier_score_cox[1][3],brier_score_one,cum_auc[0][-1]

def find_PS(Long_set,landmarking_time,all_notes,col_PS):
    ps=[]
    ps_df=pd.DataFrame()
    ps_df['Patient ID']= Long_set['Patient ID'].unique()
    for patid in ps_df['Patient ID'].values:
        subsel=all_notes[all_notes['Patient ID']==patid]
        if len(subsel)>0:
            subsel["delta idx"]=subsel["idx"]-landmarking_time
            subsel=subsel[subsel["delta idx"]<=0]
            if len(subsel)>0:
                ps.append(subsel.iloc[subsel['delta idx'].abs().argmin()][col_PS])
            else:
                ps.append(np.nan)     
        else:
            ps.append(np.nan)
    ps_df['PS_predicted']= ps
    return ps_df 

In [ ]:
#Long_set=pd.read_csv('Structured_set.csv')
#all_notes=pd.read_csv('notes_classified_semantics_v2.csv') 
coefs_all=[]
for lmw in [0,9,18,27,36]:
    coefs=[]

    landmarking_time= 27 +lmw*7 #27 days after diagnosis is median treatment start

    Static_set=Get_landmarkingset(Long_set,landmarking_time)
    ps_df=find_PS(Long_set,landmarking_time,ps_notes, 'Composite_PS')
    ps_landmarking=[ps_df[ps_df['Patient ID']==patid]['PS_predicted'].values[0] for patid in Static_set['Patient ID'].values]
    Static_set['PS']=ps_landmarking 
    print("Percentage of missing PS at landmark time: " + str(lmw))
    print(Static_set.PS.isna().sum()/len(Static_set))
    Static_set.drop(columns=['Performance Status'],inplace=True)
  
    for idx,row in Static_set.iterrows():
        if type(row['CN'])!=str:
            Static_set.loc[idx,'CN']=str(row['CN'])
    Static_set['Life Expectancy [days]']=Static_set['Life Expectancy [days]']-landmarking_time
    Static_set=Static_set[Static_set['Life Expectancy [days]']>=7]
    Surv_helper=Surv()
    Static_y=Surv_helper.from_dataframe('Vital status', 'Life Expectancy [days]', Static_set)
    Static_x=Static_set.drop(['Unnamed: 0','Time','Patient ID','Vital status','Life Expectancy [days]'],axis=1)

    cols_to_keep=['CT','CN','Morfology','Sublocalisation','Treatment','Age','Gender','PS']
    Static_x=Static_x[cols_to_keep]

    len(Static_x)

    Static_x_numeric = Static_x
    Static_y_sel=Static_y

    gb_param_grid = {"n_estimators": np.arange(1, 250, 5, dtype=int)}
    param_grid = {"alpha": 10 ** np.linspace(-4, 4, 50)} #{"alpha": 10.0 ** np.linspace(-4, 2, 100)} # 2.0 ** np.arange(-12, 13, 2)} #1np.arange(0.01,0.1,0.01)}# 
    rsf_param_grid = {"max_depth": np.arange(1, 10, dtype=int),"n_estimators":[100,300,500,800,1000]}

    N_iterations=5
    cv = ShuffleSplit(n_splits=N_iterations, test_size=1/N_iterations)
    most_imp_rsf=[]
    most_imp_rsf_noPS=[]
    
    times=[91,182,243,365,1000] #27*7]
    ##########

    #Static_temp=Static_x_numeric.replace(np.nan,999)
    cols_num=Static_x_numeric.fillna(999).columns[Static_x_numeric.fillna(999).apply(lambda s: pd.to_numeric(s, errors='coerce').notnull().all()).values]
    cols_cat=Static_x_numeric.fillna(999).columns[Static_x_numeric.fillna(999).apply(lambda s: pd.to_numeric(s, errors='coerce').notnull().all()).values==False]
    enc=OneHotEncoder().fit(Static_x_numeric[cols_cat].astype('category'))
    for z in range(0,N_iterations):

        X_train_ni, X_test_ni, y_train, y_test = train_test_split(Static_x_numeric, Static_y_sel, test_size=1/N_iterations, stratify=Static_y_sel['Vital status'],random_state=1957 + z)

        X_train_ni=Static_x_numeric.loc[X_train_ni.index]
        X_test_ni=Static_x_numeric.loc[X_test_ni.index]


        imputer= KNNImputer(n_neighbors=8, weights="uniform")
        SC=StandardScaler()
        SC.set_output(transform='pandas') 
        X_train_num=imputer.set_output(transform='pandas')
        X_train_num=imputer.fit_transform(X_train_ni[cols_num])
        X_train_num=SC.fit_transform(X_train_num)
        imputer2=SimpleImputer(strategy='most_frequent')
        X_train_cat=imputer2.set_output(transform='pandas')
        X_train_cat=imputer2.fit_transform(X_train_ni[cols_cat].replace(str(('n', 'a', 'n')),np.nan))
        X_train_cat=enc.transform(X_train_cat.astype('category'))

        imputer= KNNImputer(n_neighbors=8, weights="uniform")
        X_test_num=imputer.set_output(transform='pandas')
        X_test_num=imputer.fit_transform(X_test_ni[cols_num])
        X_test_num=SC.transform(X_test_num)
        imputer2=SimpleImputer(strategy='most_frequent')
        X_test_cat=imputer2.set_output(transform='pandas')
        X_test_cat=imputer2.fit_transform(X_test_ni[cols_cat].replace(str(('n', 'a', 'n')),np.nan))
        X_test_cat=enc.transform(X_test_cat.astype('category'))
        X_train=pd.concat([X_train_cat,X_train_num],axis=1)
        X_test=pd.concat([X_test_cat,X_test_num],axis=1)

        #Check if matrix is singular and delete columns that make it singular:
        for i in X_train.columns:
            if (X_train[i] == X_train[i].iloc[0]).all():
                #print(i)
                X_train=X_train.drop(i,axis=1)
                X_test=X_test.drop(i,axis=1)
            
        
        X_train_noPS=X_train.drop(columns=X_train.columns[X_train.columns.str.startswith('PS')].values) #['PS_regex'])
        X_test_noPS=X_test.drop(columns=X_train.columns[X_train.columns.str.startswith('PS')].values) #['PS_regex'])

        #Check if there are nans left:
        if X_train.isnull().sum().sum()>0:
            print('NaNs left')
        else:
            print('All OK!')

        print('Iteration ' + str(z) + ' RSF')

        #Max depth gridsearch optimization
        rsf = RandomSurvivalForest( min_samples_split=10, min_samples_leaf=15, n_jobs=-1)
        most_imp_rsf.append(Feature_importance(X_train,y_train,n_folds=10,False,rsf))
        
        rsf = RandomSurvivalForest( min_samples_split=10, min_samples_leaf=15, n_jobs=-1)
        rsf.fit(X_train,y_train)
        if z==0:
            Results_rsf=pd.Series(performance_metrics(rsf,y_train,y_test,times,X_train,X_test),index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months', 'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z))
        else:
            Results_rsf=pd.concat([Results_rsf,pd.Series(performance_metrics(rsf,y_train,y_test,times,X_train,X_test),index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months', 'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z))],axis=1)
        
        rsf_noPS = RandomSurvivalForest( min_samples_split=10, min_samples_leaf=15, n_jobs=-1)
        most_imp_rsf_noPS.append(Feature_importance(X_train_noPS,y_train,n_folds=10,False,rsf_noPS))
        
        rsf_noPS = RandomSurvivalForest( min_samples_split=10, min_samples_leaf=15, n_jobs=-1)
        rsf_noPS.fit(X_train_noPS,y_train)
        if z==0:
            Results_rsf_noPS=pd.Series(performance_metrics(rsf_noPS,y_train,y_test,times,X_train_noPS,X_test_noPS),index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months', 'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z))
        else:
            Results_rsf_noPS=pd.concat([Results_rsf_noPS,pd.Series(performance_metrics(rsf_noPS,y_train,y_test,times,X_train_noPS,X_test_noPS),index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months', 'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z))],axis=1)

   
    Results_rsf=pd.concat([Results_rsf,pd.Series([mean_confidence_interval3(Results_rsf.loc[i].values) for i in Results_rsf.index],index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months', 'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z+1))],axis=1)
    print(Results_rsf)
    Results_rsf_noPS=pd.concat([Results_rsf_noPS,pd.Series([mean_confidence_interval3(Results_rsf_noPS.loc[i].values) for i in Results_rsf_noPS.index],index=['C_index train test','C_index test test','C_index test test 3 months','C_index test test 6 months','C_index test test 9 months','C_index test test 12 months','Cumulative Dynamic AUC 3 months','Cumulative Dynamic AUC 6 months','Cumulative Dynamic AUC 9 months','Cumulative Dynamic AUC 12 months',                                                                                                                                                         'Brier Score 3 months', 'Brier Score 6 months', 'Brier Score 9 months', 'Brier Score 12 months', 'Brier Score', 'AUC'], name=str(z+1))],axis=1)
    print(Results_rsf_noPS)
   

    importances=[]
  
    #Results_rsf[str(z+1)].to_csv('Lanmdarking_25d' + str(lmw) +'wpstextPS_rsf.csv')
    imp_rsf=most_imp_rsf[0]
    for i in range(1,len(most_imp_rsf)):
        imp_rsf=pd.concat([imp_rsf,most_imp_rsf[i]],axis=1)
    rsf_mean=imp_rsf.mean(axis=1)
    if len(importances)>0:
        
        importances['rsf']=rsf_mean
    else:
        importances=pd.DataFrame(index=rsf_mean.index, columns=['cox','rsf'])
        importances['rsf']=rsf_mean

    #Results_rsf_noPS[str(z+1)].to_csv('Lanmdarking_25d' + str(lmw) +'wpstextnoPS_rsf.csv')
    imp_rsf_noPS=most_imp_rsf_noPS[0]
    for i in range(1,len(most_imp_rsf_noPS)):
        imp_rsf_noPS=pd.concat([imp_rsf_noPS,most_imp_rsf_noPS[i]],axis=1)
    rsf_mean_noPS=imp_rsf_noPS.mean(axis=1)
    importances['rsf no PS']=rsf_mean_noPS
   
    coefs_all.append(coefs)
    pd.concat([Results_rsf['5'],Results_rsf_noPS['5']],axis=1).to_csv('RSF' + str(lmw) + 'weeks.csv')
    importances.to_csv('RSF_importances' + str(lmw) + 'weeks.csv')
    print(pd.concat([Results_rsf['5'],Results_rsf_noPS['5']],axis=1))
    print(importances)